# **Build Self Attention from Scratch**

# Self-Attention vs Attention

## Self-Attention (Description)

Self-attention is a mechanism that allows a model to relate different positions within the same input sequence to understand contextual relationships. Instead of processing each token independently, self-attention lets every token “look at” all other tokens in the sequence and decide which ones are most relevant to it.

For each token, the model creates three vectors: **Query (Q)**, **Key (K)**, and **Value (V)**. It then compares the query of one token with the keys of all tokens to compute similarity scores. These scores determine how much attention each token should give to every other token. The final output is a weighted combination of the value vectors, where weights come from these attention scores.

---

## Attention vs Self-Attention (Learnable Parameters Comparison)

### 1. Attention (General / Encoder-Decoder Attention)

- Used when **Query comes from one sequence** and **Keys/Values come from another sequence**
- Example: Machine translation (decoder attends to encoder outputs)

**Learnable Parameters:**
- \( W_Q, W_K, W_V \)

**Key Idea:**
- Interaction between two different sequences

---

### 2. Self-Attention

- Query, Key, and Value all come from the **same sequence**
- Each token attends to every other token in the same input

**Learnable Parameters:**
- \( W_Q, W_K, W_V \)

**Key Idea:**
- Captures relationships within a single sequence

---

## Key Difference Summary

| Aspect | Attention | Self-Attention |
|--------|----------|----------------|
| Input Source | Two different sequences | Same sequence |
| Q, K, V Origin | Encoder → Decoder | Same input embedding |
| Learnable Parameters | \(W_Q, W_K, W_V\) | \(W_Q, W_K, W_V\) |
| Purpose | Cross-sequence alignment | Intra-sequence relationship modeling |

---

## Summary

Both attention and self-attention use the same learnable parameter matrices, but they differ in how inputs are used. Self-attention focuses on relationships within a single sequence, enabling models to capture context more effectively.

In [6]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

### Initializing the Head

In [1]:
import torch
import torch.nn as nn

In [ ]:
class Head(nn.Module):
    
    def __init__(self, vocab_size, embed_dim, ):
        super(Head, self).__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.query = nn.Linear(embed_dim, embed_dim, bias=False)
        self.key = nn.Linear(embed_dim, embed_dim, bias=False)
        self.value = nn.Linear(embed_dim,embed_dim, bias=False)
    
    def attention(self, x):
        embed_out = self.embeddings(x)
        
        q = self.query(embed_out)
        k = self.key(embed_out)
        v = self.value(embed_out)
        
        w = q @ k.T(-2,-1) * k.shape[-1] ** -0.5
        w = torch.nn.functional.softmax(w,dim =-1)
        
        return embed_out,w,q,k,v
    
    def forward(self, x):
        embed_out = self.embeddings(x)
        
        q = self.query(embed_out)
        k = self.key(embed_out)
        v = self.value(embed_out)
        
        #attention score
        w = q @ k.T(-2,-1) * k.shape[-1] ** -0.5
        w = torch.nn.functional.softmax(w, dim = -1)
        # Add weighted values
        out = w @ v
        
        return out    

### Dataset Defining

In [2]:
dataset = [
    (2, "PyTorch is widely used for deep learning models"),
    (1, "text classification using NLP techniques"),
    (3, "the red car was driven by him"),
    (1, "Introduction to NLP and its real world applications"),
    (3, "YOLO detects objects in real time video streams"),
    (2, "sequence models like RNNs handle time dependent data"),
    (1, "Named Entity Recognition extracts entities from text"),
    (3, "the blue car moved quickly across the road"),
    (2, "Transformers rely on attention mechanisms"),
    (1, "tokenization and embeddings are key NLP steps"),
    (3, "U-Net is used for medical image segmentation"),
    (2, "Large Language Models understand context better than RNNs"),
    (1, "sentiment analysis determines emotional tone of text"),
    (3, "a car painted red was seen on the street"),
    (2, "self attention helps models focus on relevant words in a sentence"),
]

### Tokenization

In [ ]:


tokenizor = get_tokenizer("basic_english")

#### Data Iterator

In [5]:
def yield_tokens(data):
    for _,text in data:
        yield  tokenizor(text)

#### Vocabulary

In [ ]:
vocab = build_vocab_from_iterator(yield_tokens(dataset), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])